# 🛰️ Detección de Cambios en el Terreno — Área Metropolitana de Bucaramanga

**Detección de cambios en terrenos mediante el análisis de imágenes satelitales multitemporales**

Este notebook implementa, **fase a fase**, una herramienta para detectar cambios en la cobertura
terrestre (expansión urbana, pérdida/ganancia de vegetación, variaciones en cuerpos de agua) en el
**Área Metropolitana de Bucaramanga (AMB)** — Bucaramanga, Floridablanca, Girón y Piedecuesta —,
a partir de imágenes satelitales **Sentinel-2** de acceso abierto, procesadas con **Google Earth Engine (GEE)**.

## Objetivo general
Desarrollar una herramienta para la detección de cambios en la cobertura terrestre mediante el
análisis de imágenes satelitales multitemporales, aplicando técnicas de procesamiento digital de
imágenes, con la finalidad de mejorar la eficiencia del monitoreo ambiental en un área específica.

## Estructura del notebook (una fase por cada objetivo específico)

| Fase | Objetivo específico | Contenido |
|---|---|---|
| **Fase 0** | — | Configuración del entorno (librerías y conexión a Google Earth Engine) |
| **Fase 1** | Recopilar imágenes satelitales multitemporales | Definición del área de estudio (AMB) y búsqueda de imágenes Sentinel-2 en varios periodos |
| **Fase 2** | Preprocesar las imágenes | Enmascarado de nubes, composición sin nubes e índices espectrales (NDVI, NDBI, NDWI) |
| **Fase 3** | Sistematizar la detección de cambios | Diferencias de índices, umbral estadístico reproducible y mapa de cambios clasificado |
| **Fase 4** | Evaluar y validar los resultados | Estadísticas de área, validación cruzada con datos independientes, matriz de confusión, Kappa y análisis de sensibilidad |
| **Fase 5** | Generar visualizaciones | Mapa interactivo, comparaciones antes/después, series de tiempo y exportación de resultados |

## Antes de empezar

1. **No necesitas GPU.** Todo el procesamiento pesado ocurre en los servidores de Google Earth Engine; Colab solo orquesta las solicitudes.
2. Necesitas una **cuenta de Google Earth Engine** (gratuita para uso no comercial/académico):
   - Regístrate en https://code.earthengine.google.com/register si no lo has hecho antes.
   - Crea (o reutiliza) un **proyecto de Google Cloud** asociado a Earth Engine; necesitarás su *ID de proyecto* en la Fase 0.
3. Ejecuta las celdas **en orden, de arriba hacia abajo** (▶️). Cada fase depende de las variables creadas en las fases anteriores.
4. El área de estudio y los periodos temporales son **parámetros configurables** (Fase 1): puedes ajustarlos a otra región o a otros años sin modificar el resto del notebook.
5. Al final (Fase 5) se generan y descargan: un mapa de cambios (GeoTIFF), figuras (PNG) y una tabla de estadísticas (CSV).
6. La Fase 1.6 guarda automáticamente el conjunto de imágenes recopiladas (una por periodo) en tu
   Google Drive, en `MyDrive/deteccion_cambios_bucaramanga/01_imagenes_recopiladas/` — esa carpeta
   ya existe y coincide con `CARPETA_RESULTADOS` (Fase 0.3), así que no necesitas crearla a mano.

## Fase 0. Configuración del entorno

Instalación de librerías y autenticación contra Google Earth Engine (GEE), la plataforma de acceso
abierto que usaremos como fuente de imágenes satelitales (objetivo específico 1).

### 0.1 Instalar y cargar librerías

In [ ]:
!pip install -q -U earthengine-api geemap

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import BytesIO
from PIL import Image
import os

print("Librerías cargadas correctamente.")

### 0.2 Autenticar y conectar con Google Earth Engine

Al ejecutar la siguiente celda se abrirá una ventana/enlace para iniciar sesión con tu cuenta de
Google y autorizar el acceso. Luego, reemplaza `EE_PROJECT_ID` por el **ID de tu proyecto de Google
Cloud** vinculado a Earth Engine (lo encuentras en https://code.earthengine.google.com/, arriba a la
izquierda, o en https://console.cloud.google.com/).

In [ ]:
EE_PROJECT_ID = "tu-proyecto-gee"  # <-- reemplaza con el ID de tu proyecto de Google Cloud / Earth Engine

ee.Authenticate()
ee.Initialize(project=EE_PROJECT_ID)

print("Conexión con Google Earth Engine establecida.")
print(f"Proyecto activo: {EE_PROJECT_ID}")

### 0.3 Carpeta de resultados (Google Drive)

Igual que en otros notebooks de este tipo, montamos Google Drive para guardar de forma persistente
las figuras, la tabla de estadísticas y el mapa de cambios generados en la Fase 5.

In [ ]:
from google.colab import drive

MONTAR_DRIVE = True  # pon False si no quieres usar Drive (los resultados solo quedarán en la sesión de Colab)

if MONTAR_DRIVE:
    drive.mount("/content/drive")
    CARPETA_RESULTADOS = "/content/drive/MyDrive/deteccion_cambios_bucaramanga"
else:
    CARPETA_RESULTADOS = "/content/deteccion_cambios_bucaramanga"

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
print(f"Los resultados se guardarán en: {CARPETA_RESULTADOS}")

## Fase 1. Recopilación de imágenes satelitales multitemporales

**Objetivo específico:** *Recopilar un conjunto de imágenes satelitales multitemporales de una zona
geográfica específica, mediante el uso de plataformas de acceso abierto, como fuente de análisis de
los cambios en el terreno.*

**Plataforma de acceso abierto:** [Copernicus Sentinel-2](https://sentinel.esa.int/web/sentinel/missions/sentinel-2)
(programa espacial de la Unión Europea/ESA), distribuida a través de Google Earth Engine —
colección `COPERNICUS/S2_SR_HARMONIZED` (reflectancia de superficie, ya corregida
atmosféricamente, resolución de 10 m en las bandas visibles/infrarrojo cercano).

**Zona de estudio:** Área Metropolitana de Bucaramanga (AMB) — Bucaramanga, Floridablanca, Girón y Piedecuesta.

### 1.1 Definir el área de estudio (AMB)

In [ ]:
# Rectángulo delimitador del Área Metropolitana de Bucaramanga (AMB): cubre el casco urbano
# y la franja de expansión periurbana de los 4 municipios (Bucaramanga, Floridablanca, Girón
# y Piedecuesta). Es el AOI (Area Of Interest) principal usado en todo el notebook.
AMB_LON_MIN, AMB_LON_MAX = -73.22, -73.00
AMB_LAT_MIN, AMB_LAT_MAX = 6.93, 7.20

aoi_bbox = ee.Geometry.Rectangle(
    [AMB_LON_MIN, AMB_LAT_MIN, AMB_LON_MAX, AMB_LAT_MAX]
)

# Centro aproximado del AMB (Bucaramanga), usado para centrar los mapas interactivos.
AMB_CENTRO = [7.1193, -73.1227]

area_km2 = aoi_bbox.area().divide(1e6).getInfo()
print(f"Área de estudio (rectángulo delimitador del AMB): {area_km2:,.1f} km²")

### 1.2 (Opcional) Refinar el área con límites administrativos oficiales

Esta celda intenta reemplazar el rectángulo anterior por la **unión de los límites municipales
oficiales** (Bucaramanga, Floridablanca, Girón y Piedecuesta) tomados del conjunto de datos
`FAO/GAUL/2015/level2`. Si el conjunto de datos no está disponible o los nombres no coinciden
exactamente, el notebook **conserva automáticamente** el rectángulo delimitador de la celda
anterior (`aoi_bbox`), por lo que esta celda es segura de ejecutar y no rompe el resto del flujo.

In [ ]:
MUNICIPIOS_AMB = ["Bucaramanga", "Floridablanca", "Giron", "Girón", "Piedecuesta"]

aoi = aoi_bbox  # valor por defecto: se sobrescribe abajo solo si el refinamiento funciona
aoi_fuente = "Rectángulo delimitador (definido manualmente)"

try:
    limites_municipales = (
        ee.FeatureCollection("FAO/GAUL/2015/level2")
        .filter(ee.Filter.eq("ADM1_NAME", "Santander"))
        .filter(ee.Filter.inList("ADM2_NAME", MUNICIPIOS_AMB))
    )
    n_municipios = limites_municipales.size().getInfo()
    if n_municipios > 0:
        aoi_admin = limites_municipales.geometry().dissolve()
        area_admin_km2 = aoi_admin.area().divide(1e6).getInfo()
        aoi = aoi_admin
        aoi_fuente = f"Límites administrativos oficiales (FAO GAUL, {n_municipios} municipios encontrados)"
        print(f"AOI refinada con límites oficiales: {area_admin_km2:,.1f} km²")
    else:
        print("No se encontraron los municipios por nombre en FAO GAUL; se conserva el rectángulo delimitador.")
except Exception as error:
    print(f"No fue posible refinar el AOI ({error}). Se conserva el rectángulo delimitador.")

print(f"AOI activa: {aoi_fuente}")

### 1.3 Definir los periodos multitemporales de análisis

Definimos un **conjunto de periodos** (no solo dos fechas) para cumplir con el enfoque
*multitemporal*: la comparación principal de cambios (Fases 3-4) se hará entre el primer y el
último periodo, mientras que la Fase 5 usará todos los periodos intermedios para construir una
serie de tiempo de monitoreo ambiental. Se eligen ventanas de **enero-marzo**, la temporada
relativamente más seca del año en Bucaramanga, para reducir la cobertura de nubes.

In [ ]:
# Diccionario {etiqueta: (fecha_inicio, fecha_fin)}. Puedes agregar/quitar periodos libremente;
# el resto del notebook se adapta automáticamente al primero y al último periodo del diccionario.
PERIODOS = {
    "2018": ("2018-01-01", "2018-03-31"),
    "2020": ("2020-01-01", "2020-03-31"),
    "2022": ("2022-01-01", "2022-03-31"),
    "2024": ("2024-01-01", "2024-03-31"),
}

# Porcentaje máximo de nubosidad permitido por escena (metadato de la colección).
MAX_NUBES_ESCENA = 40

ETIQUETAS_PERIODOS = list(PERIODOS.keys())
T_INICIAL_LABEL = ETIQUETAS_PERIODOS[0]
T_FINAL_LABEL = ETIQUETAS_PERIODOS[-1]

print(f"Periodos definidos: {ETIQUETAS_PERIODOS}")
print(f"Comparación principal de cambios: {T_INICIAL_LABEL} → {T_FINAL_LABEL}")

### 1.4 Recopilar las colecciones Sentinel-2 para cada periodo

In [ ]:
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"

COLECCIONES = {}

for etiqueta, (fecha_inicio, fecha_fin) in PERIODOS.items():
    coleccion = (
        ee.ImageCollection(COLECCION_S2)
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_NUBES_ESCENA))
    )
    n_imagenes = coleccion.size().getInfo()
    COLECCIONES[etiqueta] = coleccion

    estado = "✅" if n_imagenes > 0 else "⚠️ sin imágenes: amplía el rango de fechas o el % de nubes"
    print(f"{etiqueta} ({fecha_inicio} a {fecha_fin}): {n_imagenes} imágenes encontradas {estado}")

### 1.5 Vista previa rápida (color verdadero) de un periodo

In [ ]:
# Verificación visual rápida: la imagen menos nubosa del periodo final, sin ningún
# preprocesamiento todavía (eso ocurre en la Fase 2).
imagen_preliminar = COLECCIONES[T_FINAL_LABEL].sort("CLOUDY_PIXEL_PERCENTAGE").first()

vis_rgb_cruda = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}

mapa_previo = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_previo.addLayer(imagen_preliminar, vis_rgb_cruda, f"Sentinel-2 sin procesar ({T_FINAL_LABEL})")
mapa_previo.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["red"]}, "Límite del AOI (AMB)")
mapa_previo

### 1.6 Guardar en Google Drive el conjunto de imágenes recopiladas

Para poder **ver directamente en Drive** el conjunto de imágenes multitemporales recopilado (sin
depender del mapa interactivo), esta celda descarga una imagen en color verdadero por cada periodo
—la escena menos nubosa de cada uno, tal como se recopiló, sin preprocesar— y la guarda como PNG en
la carpeta ya montada en la Fase 0.3 (`CARPETA_RESULTADOS`), dentro de una subcarpeta
`01_imagenes_recopiladas`. La función `obtener_imagen_ee` se reutiliza más adelante en la Fase 5
para las figuras finales.

In [ ]:
def obtener_imagen_ee(imagen, vis_params, dimensiones=1024):
    """Descarga una imagen de Earth Engine (recortada al AOI) como objeto PIL.Image,
    lista para guardar en disco o convertir a arreglo numpy para graficar."""
    url = imagen.getThumbURL({**vis_params, "region": aoi, "dimensions": dimensiones, "format": "png"})
    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content))


CARPETA_IMAGENES_RECOPILADAS = os.path.join(CARPETA_RESULTADOS, "01_imagenes_recopiladas")
os.makedirs(CARPETA_IMAGENES_RECOPILADAS, exist_ok=True)

for etiqueta, coleccion in COLECCIONES.items():
    imagen_representativa = coleccion.sort("CLOUDY_PIXEL_PERCENTAGE").first()
    fecha_imagen = imagen_representativa.date().format("YYYY-MM-dd").getInfo()

    imagen_png = obtener_imagen_ee(imagen_representativa, vis_rgb_cruda)
    ruta_imagen = os.path.join(CARPETA_IMAGENES_RECOPILADAS, f"sentinel2_AMB_{etiqueta}_{fecha_imagen}.png")
    imagen_png.save(ruta_imagen)
    print(f"{etiqueta}: guardada {ruta_imagen} (escena del {fecha_imagen})")

print(f"\nConjunto de imágenes multitemporales guardado en Google Drive: {CARPETA_IMAGENES_RECOPILADAS}")

## Fase 2. Preprocesamiento de las imágenes satelitales

**Objetivo específico:** *Preprocesar las imágenes satelitales recopiladas, aplicando la técnica de
análisis multitemporal en la detección de cambios en el terreno, como expansión urbana,
deforestación o impacto de desastres naturales.*

Pasos: (2.1) enmascarado de nubes/sombras, (2.2) composición mediana libre de nubes por periodo,
(2.3) cálculo de índices espectrales (NDVI, NDBI, NDWI) que resumen, respectivamente, vegetación,
zonas construidas y cuerpos de agua.

### 2.1 Función de enmascarado de nubes y sombras (banda SCL)

In [ ]:
def enmascarar_nubes_sombras(imagen):
    """Enmascara nubes, sombras de nubes y cirros usando la banda SCL (Scene Classification)
    de Sentinel-2, y escala las bandas ópticas a reflectancia [0, 1]."""
    scl = imagen.select("SCL")
    # Clases SCL a excluir: 3 sombra de nube, 8 nube prob. media, 9 nube prob. alta, 10 cirro delgado.
    mascara_valida = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))

    bandas_opticas = imagen.select("B.*").multiply(0.0001)
    return (
        bandas_opticas.updateMask(mascara_valida)
        .copyProperties(imagen, imagen.propertyNames())
    )

print("Función de enmascarado de nubes definida.")

### 2.2 Generar el composite mediano (sin nubes) de cada periodo

In [ ]:
COMPOSITES = {}

for etiqueta, coleccion in COLECCIONES.items():
    composite = (
        coleccion
        .map(enmascarar_nubes_sombras)
        .median()
        .clip(aoi)
    )
    COMPOSITES[etiqueta] = composite

print(f"Composites generados para los periodos: {list(COMPOSITES.keys())}")

### 2.3 Cálculo de índices espectrales

In [ ]:
def calcular_indices(imagen):
    """Agrega al composite las bandas de índices espectrales:
    - NDVI  (B8, B4): vegetación
    - NDBI  (B11, B8): superficies construidas / suelo urbano
    - NDWI  (B3, B8): cuerpos de agua (McFeeters, 1996)
    """
    ndvi = imagen.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndbi = imagen.normalizedDifference(["B11", "B8"]).rename("NDBI")
    ndwi = imagen.normalizedDifference(["B3", "B8"]).rename("NDWI")
    return imagen.addBands([ndvi, ndbi, ndwi])

COMPOSITES = {etiqueta: calcular_indices(img) for etiqueta, img in COMPOSITES.items()}

print("Índices NDVI, NDBI y NDWI calculados para cada periodo.")
print("Bandas disponibles en cada composite:", COMPOSITES[T_FINAL_LABEL].bandNames().getInfo())

### 2.4 Verificación visual del preprocesamiento

In [ ]:
vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}
vis_ndvi = {"bands": ["NDVI"], "min": -0.2, "max": 0.8, "palette": ["#a50026", "#ffffbf", "#1a9850"]}

mapa_preprocesado = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_preprocesado.addLayer(COMPOSITES[T_INICIAL_LABEL], vis_rgb, f"Color verdadero {T_INICIAL_LABEL} (sin nubes)")
mapa_preprocesado.addLayer(COMPOSITES[T_FINAL_LABEL], vis_rgb, f"Color verdadero {T_FINAL_LABEL} (sin nubes)")
mapa_preprocesado.addLayer(COMPOSITES[T_INICIAL_LABEL], vis_ndvi, f"NDVI {T_INICIAL_LABEL}", shown=False)
mapa_preprocesado.addLayer(COMPOSITES[T_FINAL_LABEL], vis_ndvi, f"NDVI {T_FINAL_LABEL}", shown=False)
mapa_preprocesado.addLayerControl()
mapa_preprocesado

## Fase 3. Sistematización de la detección de cambios

**Objetivo específico:** *Sistematizar la técnica de detección de los cambios en la cobertura
terrestre, identificando las variaciones significativas en el terreno especificado.*

Metodología (reproducible, no basada en inspección visual subjetiva):

1. Diferencia de índices espectrales entre el periodo inicial y el final (ΔNDVI, ΔNDBI, ΔNDWI).
2. **Umbral estadístico sistemático**: media ± *k* × desviación estándar de cada diferencia,
   calculado sobre el propio AOI — el mismo criterio se puede aplicar a cualquier otra zona o par
   de fechas sin ajuste manual.
3. Combinación de los tres índices en un **mapa de cambios clasificado** con categorías
   interpretables (expansión urbana, pérdida/ganancia de vegetación, cambio hídrico, sin cambio).

### 3.1 Composites del periodo inicial y final

In [ ]:
imagen_inicial = COMPOSITES[T_INICIAL_LABEL]
imagen_final = COMPOSITES[T_FINAL_LABEL]

print(f"Periodo inicial: {T_INICIAL_LABEL} ({PERIODOS[T_INICIAL_LABEL][0]} a {PERIODOS[T_INICIAL_LABEL][1]})")
print(f"Periodo final:   {T_FINAL_LABEL} ({PERIODOS[T_FINAL_LABEL][0]} a {PERIODOS[T_FINAL_LABEL][1]})")

### 3.2 Diferencia de índices espectrales (ΔNDVI, ΔNDBI, ΔNDWI)

In [ ]:
delta_ndvi = imagen_final.select("NDVI").subtract(imagen_inicial.select("NDVI")).rename("delta_NDVI")
delta_ndbi = imagen_final.select("NDBI").subtract(imagen_inicial.select("NDBI")).rename("delta_NDBI")
delta_ndwi = imagen_final.select("NDWI").subtract(imagen_inicial.select("NDWI")).rename("delta_NDWI")

deltas = delta_ndvi.addBands([delta_ndbi, delta_ndwi])
print("Bandas de diferencia calculadas:", deltas.bandNames().getInfo())

### 3.3 Umbral estadístico sistemático (media ± k·desviación estándar)

In [ ]:
K_UMBRAL = 1.5  # sensibilidad del umbral; se explora en la Fase 4.4 (análisis de sensibilidad)

def obtener_umbrales(imagen_delta, banda, aoi, escala=20, k=K_UMBRAL):
    """Calcula (umbral_inferior, umbral_superior) = media ± k·desv. estándar de una banda
    de diferencia, sobre el AOI. Es el mismo criterio estadístico para cualquier banda/zona."""
    estadisticas = imagen_delta.select(banda).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi,
        scale=escala,
        bestEffort=True,
        maxPixels=1e9,
    ).getInfo()

    media = estadisticas[f"{banda}_mean"]
    desviacion = estadisticas[f"{banda}_stdDev"]
    return media - k * desviacion, media + k * desviacion

umbral_ndvi_inf, umbral_ndvi_sup = obtener_umbrales(deltas, "delta_NDVI", aoi)
umbral_ndbi_inf, umbral_ndbi_sup = obtener_umbrales(deltas, "delta_NDBI", aoi)
umbral_ndwi_inf, umbral_ndwi_sup = obtener_umbrales(deltas, "delta_NDWI", aoi)

print(f"Umbral ΔNDVI: [{umbral_ndvi_inf:.4f}, {umbral_ndvi_sup:.4f}]  (pérdida de vegetación / ganancia de vegetación)")
print(f"Umbral ΔNDBI: [{umbral_ndbi_inf:.4f}, {umbral_ndbi_sup:.4f}]  (expansión urbana)")
print(f"Umbral ΔNDWI: [{umbral_ndwi_inf:.4f}, {umbral_ndwi_sup:.4f}]  (cambio hídrico)")

### 3.4 Análisis de Vector de Cambio (CVA) — magnitud del cambio

Complementa el análisis por índices individuales con una medida conjunta de "cuánto" cambió cada
píxel, combinando vegetación (NDVI) y superficie construida (NDBI) en un solo vector.

In [ ]:
magnitud_cambio = (
    deltas.select("delta_NDVI").pow(2)
    .add(deltas.select("delta_NDBI").pow(2))
    .sqrt()
    .rename("magnitud_CVA")
)

umbral_magnitud = magnitud_cambio.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9,
).getInfo()

umbral_cva = umbral_magnitud["magnitud_CVA_mean"] + K_UMBRAL * umbral_magnitud["magnitud_CVA_stdDev"]
print(f"Umbral de magnitud de cambio (CVA): {umbral_cva:.4f} — píxeles por encima se consideran 'cambio relevante'")

### 3.5 Mapa de cambios clasificado

Se construye una única capa con categorías mutuamente excluyentes, aplicando los umbrales
definidos arriba en orden de prioridad (agua → vegetación → urbano → sin cambio):

| Código | Categoría | Criterio |
|---|---|---|
| 0 | Sin cambio significativo | ninguno de los criterios siguientes se cumple |
| 1 | Pérdida de vegetación (posible deforestación) | ΔNDVI < umbral inferior |
| 2 | Ganancia de vegetación | ΔNDVI > umbral superior |
| 3 | Expansión urbana / suelo construido | ΔNDBI > umbral superior |
| 4 | Cambio en cuerpos de agua | \|ΔNDWI\| > umbral superior |

In [ ]:
sin_cambio = ee.Image(0)

mapa_cambios = (
    sin_cambio
    .where(deltas.select("delta_NDVI").gt(umbral_ndvi_sup), 2)   # ganancia de vegetación
    .where(deltas.select("delta_NDVI").lt(umbral_ndvi_inf), 1)   # pérdida de vegetación
    .where(deltas.select("delta_NDBI").gt(umbral_ndbi_sup), 3)   # expansión urbana
    .where(deltas.select("delta_NDWI").abs().gt(umbral_ndwi_sup), 4)  # cambio hídrico
    .rename("clase_cambio")
    .clip(aoi)
    .updateMask(imagen_inicial.select("NDVI").mask().And(imagen_final.select("NDVI").mask()))
)

CATEGORIAS_CAMBIO = {
    0: ("Sin cambio significativo", "#d9d9d9"),
    1: ("Pérdida de vegetación (posible deforestación)", "#d73027"),
    2: ("Ganancia de vegetación", "#1a9850"),
    3: ("Expansión urbana / suelo construido", "#fdae61"),
    4: ("Cambio en cuerpos de agua", "#4575b4"),
}

paleta_cambios = [color for _, color in CATEGORIAS_CAMBIO.values()]

mapa_clasificado_vis = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_clasificado_vis.addLayer(
    mapa_cambios,
    {"min": 0, "max": 4, "palette": paleta_cambios},
    f"Cambios {T_INICIAL_LABEL} → {T_FINAL_LABEL}",
)
mapa_clasificado_vis.add_legend(
    title="Categoría de cambio",
    legend_dict={nombre: color for nombre, color in CATEGORIAS_CAMBIO.values()},
)
mapa_clasificado_vis

## Fase 4. Evaluación y validación de los resultados

**Objetivo específico:** *Evaluar los resultados obtenidos, validando la confiabilidad de la
herramienta propuesta, mediante el análisis comparativo de las imágenes satelitales multitemporales.*

Se valida el mapa de cambios de tres formas complementarias:
1. Estadísticas de área por categoría (magnitud del fenómeno detectado).
2. Comparación con un conjunto de datos **independiente** (Dynamic World de Google) mediante
   matriz de confusión e índice **Kappa**.
3. Análisis de sensibilidad del umbral estadístico (robustez del método ante *k*).

### 4.1 Estadísticas de área por categoría de cambio

In [ ]:
area_pixel = ee.Image.pixelArea().divide(10000)  # hectáreas por píxel

estadisticas_area = (
    area_pixel.addBands(mapa_cambios)
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="clase"),
        geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9,
    )
    .getInfo()
)

filas = []
area_total_ha = 0
for grupo in estadisticas_area["groups"]:
    clase = int(grupo["clase"])
    hectareas = grupo["sum"]
    area_total_ha += hectareas
    nombre, color = CATEGORIAS_CAMBIO[clase]
    filas.append({"clase": clase, "categoria": nombre, "hectareas": hectareas, "color": color})

tabla_area = pd.DataFrame(filas).sort_values("clase").reset_index(drop=True)
tabla_area["porcentaje"] = 100 * tabla_area["hectareas"] / area_total_ha

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
tabla_area[["categoria", "hectareas", "porcentaje"]]

### 4.2 Validación cruzada con un conjunto de datos independiente (Dynamic World)

No se cuenta con datos de verdad de campo, por lo que la validación se hace comparando el mapa de
cambios propio contra un producto de cobertura de suelo **independiente y ya publicado**:
[Dynamic World](https://dynamicworld.app/) (Google/WRI), que ofrece una probabilidad de
"superficie construida" (`built`) casi en tiempo real desde 2015, a 10 m de resolución. Se deriva
un cambio de urbanización a partir de Dynamic World y se compara contra la categoría "Expansión
urbana" (clase 3) de nuestro mapa.

In [ ]:
def obtener_built_dw(fecha_inicio, fecha_fin, aoi):
    coleccion_dw = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        .select("built")
    )
    return coleccion_dw.median().clip(aoi)

built_inicial = obtener_built_dw(*PERIODOS[T_INICIAL_LABEL], aoi)
built_final = obtener_built_dw(*PERIODOS[T_FINAL_LABEL], aoi)
delta_built_dw = built_final.subtract(built_inicial).rename("delta_built")

UMBRAL_EXPANSION_DW = 0.10  # aumento mínimo de probabilidad "built" para considerarlo expansión urbana
expansion_urbana_dw = delta_built_dw.gt(UMBRAL_EXPANSION_DW).rename("dw_expansion")
expansion_urbana_propia = mapa_cambios.eq(3).rename("propia_expansion")

print("Capas de referencia (Dynamic World) y propias listas para comparar.")

### 4.3 Muestreo aleatorio estratificado, matriz de confusión e índice Kappa

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score, classification_report

capas_comparacion = expansion_urbana_propia.addBands(expansion_urbana_dw)

muestra = capas_comparacion.stratifiedSample(
    numPoints=150,
    classBand="propia_expansion",
    region=aoi,
    scale=20,
    seed=42,
    geometries=False,
).getInfo()

registros = [f["properties"] for f in muestra["features"]]
df_muestra = pd.DataFrame(registros).dropna()

y_propio = df_muestra["propia_expansion"].astype(int)
y_referencia = df_muestra["dw_expansion"].astype(int)

matriz_confusion = confusion_matrix(y_referencia, y_propio, labels=[0, 1])
kappa = cohen_kappa_score(y_referencia, y_propio)
exactitud_global = (y_referencia == y_propio).mean()

print(f"Tamaño de la muestra evaluada: {len(df_muestra)} puntos")
print(f"Exactitud global (vs. Dynamic World): {exactitud_global:.1%}")
print(f"Índice Kappa de Cohen: {kappa:.3f}")
print("\nMatriz de confusión (filas = referencia Dynamic World, columnas = mapa propio):")
print(pd.DataFrame(
    matriz_confusion,
    index=["Ref: sin expansión", "Ref: expansión"],
    columns=["Propio: sin expansión", "Propio: expansión"],
))
print("\nReporte de clasificación:")
print(classification_report(y_referencia, y_propio, target_names=["sin expansión", "expansión urbana"]))

### 4.4 Análisis de sensibilidad del umbral (robustez del método)

Se recalcula el porcentaje de área con cambios significativos usando distintos valores de *k*
(multiplicador de la desviación estándar) para verificar que la herramienta no depende de forma
crítica de un único valor arbitrario.

In [ ]:
valores_k = [1.0, 1.25, 1.5, 1.75, 2.0]
resultados_sensibilidad = []

for k in valores_k:
    umb_ndvi_inf, umb_ndvi_sup = obtener_umbrales(deltas, "delta_NDVI", aoi, k=k)
    umb_ndbi_inf, umb_ndbi_sup = obtener_umbrales(deltas, "delta_NDBI", aoi, k=k)

    mapa_k = (
        ee.Image(0)
        .where(deltas.select("delta_NDVI").gt(umb_ndvi_sup), 2)
        .where(deltas.select("delta_NDVI").lt(umb_ndvi_inf), 1)
        .where(deltas.select("delta_NDBI").gt(umb_ndbi_sup), 3)
    )

    area_cambio_km2 = (
        ee.Image.pixelArea().divide(1e6)
        .updateMask(mapa_k.gt(0))
        .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9)
        .getInfo()
    )
    km2 = area_cambio_km2.get("area", 0)
    resultados_sensibilidad.append({"k": k, "area_cambio_km2": km2, "pct_area_total": 100 * km2 / area_km2})

tabla_sensibilidad = pd.DataFrame(resultados_sensibilidad)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tabla_sensibilidad["k"], tabla_sensibilidad["pct_area_total"], marker="o", color="#2166ac")
ax.set_xlabel("k (umbral = media ± k·desviación estándar)")
ax.set_ylabel("% del AOI clasificado como cambio")
ax.set_title("Análisis de sensibilidad del umbral estadístico")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "sensibilidad_umbral.png"), dpi=150)
plt.show()

tabla_sensibilidad

## Fase 5. Visualización de los resultados

**Objetivo específico:** *Generar visualizaciones de los cambios detectados, empleando librerías
de representación gráfica, permitiendo la interpretación de los cambios detectados.*

### 5.1 Mapa interactivo comparativo (antes / después / cambios)

In [ ]:
mapa_final = geemap.Map(center=AMB_CENTRO, zoom=11)
mapa_final.addLayer(imagen_inicial, vis_rgb, f"Color verdadero {T_INICIAL_LABEL}", shown=False)
mapa_final.addLayer(imagen_final, vis_rgb, f"Color verdadero {T_FINAL_LABEL}", shown=True)
mapa_final.addLayer(
    mapa_cambios,
    {"min": 0, "max": 4, "palette": paleta_cambios},
    f"Mapa de cambios {T_INICIAL_LABEL} → {T_FINAL_LABEL}",
)
mapa_final.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["black"]}, "Límite del AOI (AMB)")
mapa_final.add_legend(
    title="Categoría de cambio",
    legend_dict={nombre: color for nombre, color in CATEGORIAS_CAMBIO.values()},
)
mapa_final.addLayerControl()
mapa_final

### 5.2 Comparación visual estática (antes / después / cambios)

Reutiliza `obtener_imagen_ee` (definida en la Fase 1.6) para descargar las miniaturas como arreglos
numpy y componer la figura comparativa.

In [ ]:
img_inicial_arr = np.array(obtener_imagen_ee(imagen_inicial, vis_rgb))
img_final_arr = np.array(obtener_imagen_ee(imagen_final, vis_rgb))
img_cambios_arr = np.array(obtener_imagen_ee(mapa_cambios, {"min": 0, "max": 4, "palette": paleta_cambios}))

fig, ejes = plt.subplots(1, 3, figsize=(16, 5.5))

ejes[0].imshow(img_inicial_arr)
ejes[0].set_title(f"Área Metropolitana de Bucaramanga — {T_INICIAL_LABEL}")
ejes[0].axis("off")

ejes[1].imshow(img_final_arr)
ejes[1].set_title(f"Área Metropolitana de Bucaramanga — {T_FINAL_LABEL}")
ejes[1].axis("off")

ejes[2].imshow(img_cambios_arr)
ejes[2].set_title(f"Cambios detectados ({T_INICIAL_LABEL} → {T_FINAL_LABEL})")
ejes[2].axis("off")

parches_leyenda = [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=color, markersize=12, label=nombre)
    for nombre, color in CATEGORIAS_CAMBIO.values()
]
fig.legend(handles=parches_leyenda, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "comparacion_antes_despues_cambios.png"), dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Serie de tiempo de NDVI y NDBI promedio en el AMB (monitoreo multitemporal)

In [ ]:
registros_series = []
for etiqueta in ETIQUETAS_PERIODOS:
    promedios = COMPOSITES[etiqueta].select(["NDVI", "NDBI"]).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, bestEffort=True, maxPixels=1e9,
    ).getInfo()
    registros_series.append({"periodo": etiqueta, "NDVI_promedio": promedios["NDVI"], "NDBI_promedio": promedios["NDBI"]})

serie_tiempo = pd.DataFrame(registros_series)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(serie_tiempo["periodo"], serie_tiempo["NDVI_promedio"], marker="o", color="#1a9850", label="NDVI promedio (vegetación)")
ax.plot(serie_tiempo["periodo"], serie_tiempo["NDBI_promedio"], marker="s", color="#fdae61", label="NDBI promedio (suelo construido)")
ax.set_xlabel("Periodo")
ax.set_ylabel("Valor promedio del índice")
ax.set_title("Evolución multitemporal de NDVI y NDBI en el AMB")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "serie_tiempo_ndvi_ndbi.png"), dpi=150)
plt.show()

serie_tiempo

### 5.4 Gráfico de área por categoría de cambio

In [ ]:
tabla_area_graf = tabla_area[tabla_area["clase"] != 0].sort_values("hectareas", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.barh(tabla_area_graf["categoria"], tabla_area_graf["hectareas"], color=tabla_area_graf["color"])
ax.set_xlabel("Área (hectáreas)")
ax.set_title(f"Área por categoría de cambio — AMB ({T_INICIAL_LABEL} → {T_FINAL_LABEL})")
for barra, valor in zip(barras, tabla_area_graf["hectareas"]):
    ax.text(valor, barra.get_y() + barra.get_height() / 2, f" {valor:,.0f} ha", va="center")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "area_por_categoria.png"), dpi=150)
plt.show()

### 5.5 Exportar resultados

Se guardan/descargan tres productos finales:
1. **Tabla de estadísticas** (`estadisticas_cambio.csv`) — Fase 4.1.
2. **Figuras** (PNG) generadas en 4.4 y 5.2-5.4 — ya guardadas en `CARPETA_RESULTADOS`.
3. **Mapa de cambios clasificado** (GeoTIFF) — exportado a Google Drive mediante una tarea de
   Earth Engine (puede tardar algunos minutos; revisa el progreso en la pestaña *Tasks* de
   https://code.earthengine.google.com/).

In [ ]:
ruta_csv = os.path.join(CARPETA_RESULTADOS, "estadisticas_cambio.csv")
tabla_area.to_csv(ruta_csv, index=False)
print(f"Tabla de estadísticas guardada en: {ruta_csv}")

tarea_exportacion = ee.batch.Export.image.toDrive(
    image=mapa_cambios.toByte(),
    description="mapa_cambios_AMB",
    folder="deteccion_cambios_bucaramanga",
    fileNamePrefix=f"mapa_cambios_{T_INICIAL_LABEL}_{T_FINAL_LABEL}",
    region=aoi,
    scale=10,
    maxPixels=1e9,
)
tarea_exportacion.start()
print("Tarea de exportación del mapa de cambios (GeoTIFF) iniciada.")
print("Revisa su progreso en la pestaña 'Tasks' de https://code.earthengine.google.com/")

from google.colab import files
files.download(ruta_csv)

## Conclusiones y próximos pasos

Este notebook cubrió, fase a fase, los cinco objetivos específicos del proyecto para el Área
Metropolitana de Bucaramanga:

1. **Recopilación** — Imágenes Sentinel-2 de acceso abierto (Google Earth Engine) para varios periodos.
2. **Preprocesamiento** — Enmascarado de nubes, composición mediana e índices espectrales (NDVI, NDBI, NDWI).
3. **Sistematización** — Diferencias de índices con umbral estadístico reproducible y mapa de cambios clasificado.
4. **Evaluación** — Estadísticas de área, validación cruzada con Dynamic World (Kappa/matriz de confusión) y análisis de sensibilidad del umbral.
5. **Visualización** — Mapa interactivo, comparación antes/después, serie de tiempo y exportación de resultados.

**Posibles extensiones:**
- Incorporar imágenes de mayor resolución (p. ej. PlanetScope) para detectar cambios más finos.
- Añadir más periodos a `PERIODOS` para un monitoreo continuo (mensual/anual).
- Sustituir la clasificación basada en umbrales por un clasificador supervisado (Random Forest /
  redes neuronales) entrenado con puntos de verdad de campo, si llegan a estar disponibles.
- Aplicar la misma metodología a otras zonas cambiando únicamente `aoi_bbox` y `MUNICIPIOS_AMB` (Fase 1).

## Solución de problemas

- **`EEException: Not signed up for Earth Engine` o error de autenticación**: crea/activa tu cuenta en
  https://code.earthengine.google.com/register y verifica que `EE_PROJECT_ID` (Fase 0.2) sea el ID
  correcto de tu proyecto de Google Cloud.
- **`0 imágenes encontradas` en algún periodo (Fase 1.4)**: amplía el rango de fechas del periodo en
  `PERIODOS` o aumenta `MAX_NUBES_ESCENA`; Bucaramanga tiene temporadas con alta nubosidad.
- **`Computation timed out` o `Too many pixels in the region`**: reduce el parámetro `scale` en las
  llamadas a `reduceRegion` (por ejemplo de 10 a 20 o 30 m) o reduce el tamaño del AOI.
- **La celda 1.2 imprime "No fue posible refinar el AOI"**: no es un error crítico — el notebook
  sigue funcionando con el rectángulo delimitador definido en la celda 1.1.
- **La descarga de miniaturas (`descargar_thumbnail`, Fase 5.2) falla o tarda mucho**: revisa tu
  conexión a internet; si el AOI es muy grande, reduce el parámetro `dimensiones`.
- **El Kappa de la Fase 4.3 sale muy bajo**: es esperable cierto desacuerdo, ya que Dynamic World y
  el método propio usan sensores/criterios distintos; revisa `UMBRAL_EXPANSION_DW` y `K_UMBRAL` — el
  análisis de sensibilidad (4.4) ayuda a elegir un valor más estable.